# 06b Cleaner contours, satellite and side by side

Phase 6b check, after your 06a notes: contours are now generalised like printed maps, `basemap-satellite` and `basemap-hybrid` add imagery, and `kh.compare` shows maps side by side, panning and zooming together. Needs internet for the elevation and imagery tiles. Record pass or fail in the table at the end.

In [1]:
import warnings
import webbrowser
from pathlib import Path

import geopandas as gpd

import kharita as kh

warnings.filterwarnings("ignore", module="lonboard")
roads = gpd.read_file("../data/processed/khartoum_roads.gpkg")
TERRAIN = "Elevation: Mapzen Terrain Tiles on AWS Open Data."
MARRA = (24.0, 12.75, 24.5, 13.15)
LIGHT = "stroke-amber-200/90 width-[weight] width-max-1.6 legend-none"

In [2]:
# A. Detail levels: high keeps more bumps, low keeps only the main shapes (drag one map, both move)
high = kh.contours(MARRA, interval=100, detail="high")
low = kh.contours(MARRA, interval=100, detail="low")
marra = kh.contours(MARRA, interval=100)
print("lines: high", len(high), "medium", len(marra), "low", len(low), "| blur, m:", marra.attrs["smooth"])
kh.compare(
    kh.map(high, "preset-topo basemap-topo", title="detail high"),
    kh.map(low, "preset-topo basemap-topo", title="detail low"),
)

lines: high 96 medium 47 low 29 | blur, m: 300


GridBox(children=(VBox(children=(HTML(value='<style>.kh-overlay-tl{z-index:10;pointer-events:auto;align-self:s…

In [3]:
# B. Your request: the topographic map next to the same contours on satellite imagery
index = marra[marra["index"]]
topo = kh.Map(title="Jebel Marra", subtitle="Contours every 100 m", source=TERRAIN)
topo.add(marra, "preset-topo basemap-topo")
topo.add(index, "stroke-transparent label-[elevation] legend-none")
sat = kh.Map(title="Jebel Marra from space", subtitle="Sentinel-2 cloudless", source=TERRAIN)
sat.add(marra, LIGHT + " basemap-hybrid")
sat.add(index, "stroke-transparent label-[elevation] legend-none")
kh.compare(topo, sat)

GridBox(children=(VBox(children=(HTML(value='<style>.kh-overlay-tl{z-index:10;pointer-events:auto;align-self:s…

In [4]:
# C. Phase 5 relief designs under the contours: tinted relief, and sharp relief coloured by height
kh.compare(
    kh.map(marra, "preset-topo basemap-paper relief-tint", title="Tint and contours"),
    kh.map(marra, "basemap-midnight relief-sharp stroke-[elevation] ramp-thermal scheme-linear width-[weight] width-max-1.8",
           legend={"title": "Elevation, m", "fmt": "{:,.0f}"}, title="Sharp relief, coloured"),
)

GridBox(children=(VBox(children=(HTML(value='<style>.kh-overlay-tl{z-index:10;pointer-events:auto;align-self:s…

In [5]:
# D. Imagery, shaded relief and contours together on one map
kh.map(marra, LIGHT + " basemap-satellite relief-soft", title="Imagery, relief and contours", source=TERRAIN)

In [6]:
# E. The Red Sea hills again: stepped relief against hybrid imagery
hills = kh.contours((36.0, 18.6, 37.4, 19.8))
print(len(hills), "lines, interval", hills.attrs["interval"], "m, zoom", hills.attrs["zoom"], "blur", hills.attrs["smooth"], "m")
kh.compare(
    kh.map(hills, "preset-topo basemap-topo relief-steps", title="Red Sea hills"),
    kh.map(hills, LIGHT + " basemap-hybrid", title="Red Sea hills, imagery"),
)

194 lines, interval 100 m, zoom 9 blur 491 m


GridBox(children=(VBox(children=(HTML(value='<style>.kh-overlay-tl{z-index:10;pointer-events:auto;align-self:s…

In [7]:
# F. City data on imagery: Khartoum roads by rank over hybrid, next to muted
style = "stroke-[road_class] ramp-set2 width-[rank] width-max-4 legend-none"
kh.compare(
    kh.map(roads, style + " basemap-muted", title="Muted"),
    kh.map(roads, style + " basemap-satellite", title="Satellite"),
)

GridBox(children=(VBox(children=(HTML(value='<style>.kh-overlay-tl{z-index:10;pointer-events:auto;align-self:s…

In [8]:
# G. Newer imagery (2024 is free for non-commercial use only); back to 2016 afterwards
old = kh.map(roads, "stroke-white/70 basemap-satellite legend-none", title="2016")
_ = old.lonboard
kh.imagery(2024)
new = kh.map(roads, "stroke-white/70 basemap-satellite legend-none", title="2024")
_ = new.lonboard
kh.imagery(2016)
kh.compare(old, new)

GridBox(children=(VBox(children=(HTML(value='<style>.kh-overlay-tl{z-index:10;pointer-events:auto;align-self:s…

In [9]:
# H. Export the imagery map with contours
out = Path("../spike_out")
out.mkdir(exist_ok=True)
sat.to_html(out / "jebel_marra_satellite.html")
webbrowser.open((out / "jebel_marra_satellite.html").resolve().as_uri())

True

## Results

| Test | What passing looks like | Pass? | Notes |
| --- | --- | --- | --- |
| A | Both maps clean, no tiny loops or jagged edges; low simpler than high; dragging one moves the other | | |
| B | Topographic map left, imagery with light contours and white height labels right | | |
| C | Contours over tinted relief left; coloured contours over sharp relief right | | |
| D | Imagery shaded by relief with contours on top | | |
| E | Red Sea hills readable on both; far fewer lines than 06a (662) | | |
| F | Roads over muted and over satellite; satellite map has no flat building blocks | | |
| G | Two imagery years side by side; 2024 sharper or differently coloured | | |
| H | Exported page matches B right | | |